# P5 – Control de Trayectorias — MyCobot 280

**Objetivos:**
1. Definir las poses clave: `init_pose`, `watch_pose`, `pick_pose`, `place_pose`
2. Implementar el ciclo de agarre con `send_angles` / `send_coords` + `set_gripper_value`
3. Calibrar velocidades y tiempos de espera
4. Ejecutar 5 ciclos consecutivos sin intervención humana y registrar tasa de éxito
5. Gestionar errores de comunicación con reintentos y timeouts

## 0. Conexión al robot

In [ ]:
import time
import numpy as np
from datetime import datetime
from pymycobot.mycobot import MyCobot

PORT = '/dev/ttyUSB0'
BAUD = 1000000

mc = MyCobot(PORT, BAUD)
mc.power_on()
time.sleep(1)

print(f'Conectado en {PORT} a {BAUD} baud')
print(f'Versión firmware: {mc.get_system_version()}')

## 1. Poses clave del ciclo

Cada pose es un vector `[q1, q2, q3, q4, q5, q6]` en grados (ángulos de hardware).

| Pose | Descripción |
|------|-------------|
| `init_pose` | Posición de reposo segura al inicio y fin de cada ciclo |
| `watch_pose` | Cámara apuntando a la zona de trabajo para detectar el objeto |
| `pick_pose` | Sobre el objeto antes de bajar a agarrar |
| `place_pose` | Zona de depósito donde se suelta el objeto |

In [ ]:

# ---------------------------------------------------------------
# Sección 1: Poses clave — tomadas del robot real (ctrl_joints)
# ---------------------------------------------------------------

# Poses de la trayectoria A -> B (transportar objeto)
PICK_UPPER  = [121.2,  -9.58,  -66.53, -11.07, -3.86,  70.75]  # sobre objeto A
PICK_GRASP  = [120.32, -31.72, -92.02,  34.27, -3.07,  69.78]  # bajar a A (agarre)
PLACE_UPPER = [-83.67, -14.41, -72.33,  -1.4,  -2.46,  44.20]  # sobre posición B
PLACE_GRASP = [-82.61, -24.87, -98.52,  38.23, -3.16,  44.29]  # bajar a B (depósito)
HOME        = [0, 0, 0, 0, 0, 0]                                # posición de reposo

# Parámetros
MOVE_SPEED    = 90    # igual que en ctrl_joints
GRIPPER_SPEED = 50
WAIT_MOVE     = 1.5   # segundos tras send_angles
WAIT_GRIP     = 1.0   # segundos tras set_gripper_state

# Muestra resumen de poses
print('Poses del ciclo A → B:')
print(f'  HOME        : {HOME}')
print(f'  PICK_UPPER  : {PICK_UPPER}')
print(f'  PICK_GRASP  : {PICK_GRASP}')
print(f'  PLACE_UPPER : {PLACE_UPPER}')
print(f'  PLACE_GRASP : {PLACE_GRASP}')


### 1.1 Capturar poses desde el robot

Mueve el robot manualmente a la posición deseada y ejecuta la celda para leer los ángulos reales.

In [ ]:
# ---------------------------------------------------------------
# Lee la pose actual y la guarda en el diccionario POSES
# Cambia POSE_A_CAPTURAR al nombre que quieras calibrar
# ---------------------------------------------------------------
POSE_A_CAPTURAR = 'watch_pose'   # <-- cambiar según necesidad

time.sleep(0.3)
angulos_actuales = mc.get_angles()
coords_actuales  = mc.get_coords()
time.sleep(0.2)

POSES[POSE_A_CAPTURAR] = list(angulos_actuales)

print(f'Pose capturada -> {POSE_A_CAPTURAR}')
print(f'  Ángulos : {[round(a, 2) for a in angulos_actuales]}')
if coords_actuales:
    print(f'  Coords  : x={coords_actuales[0]:.1f}  y={coords_actuales[1]:.1f}  z={coords_actuales[2]:.1f} mm')

## 2. Funciones de movimiento con manejo de errores

Sección 5 del enunciado: reintentos y timeouts ante fallos de comunicación USB/serial.

In [ ]:

# ---------------------------------------------------------------
# Sección 5: Gestión de errores — reintentos y timeouts
# ---------------------------------------------------------------
MAX_RETRIES = 3
RETRY_DELAY = 1.0

def send_with_retry(func, *args, retries=MAX_RETRIES):
    for intento in range(1, retries + 1):
        try:
            return func(*args)
        except Exception as e:
            print(f'  [REINTENTO {intento}/{retries}] {func.__name__}: {e}')
            time.sleep(RETRY_DELAY)
    raise RuntimeError(f'Fallo tras {retries} intentos en {func.__name__}')


def ejecutar_secuencia(poses):
    """
    Ejecuta una lista de poses donde cada elemento es:
      - list  → mc.send_angles(pose, MOVE_SPEED)
      - 'open'  → mc.set_gripper_state(0, GRIPPER_SPEED)
      - 'close' → mc.set_gripper_state(1, GRIPPER_SPEED)
    Igual al patron usado en ctrl_joints.ipynb del JetCobot.
    """
    for paso, pose in enumerate(poses):
        if pose == 'close':
            print(f'    paso {paso+1}: CERRAR gripper')
            send_with_retry(mc.set_gripper_state, 1, GRIPPER_SPEED)
            time.sleep(WAIT_GRIP)
        elif pose == 'open':
            print(f'    paso {paso+1}: ABRIR gripper')
            send_with_retry(mc.set_gripper_state, 0, GRIPPER_SPEED)
            time.sleep(WAIT_GRIP)
        else:
            print(f'    paso {paso+1}: send_angles {[round(a,2) for a in pose]}')
            send_with_retry(mc.send_angles, pose, MOVE_SPEED)
            time.sleep(WAIT_MOVE)


print('Funciones de movimiento definidas.')


## 3. Calibración de velocidades y tiempos de espera

Ejecuta esta celda para probar distintas velocidades y medir el tiempo real de movimiento.

In [ ]:
# ---------------------------------------------------------------
# Sección 3: Calibración de velocidades
# Prueba init_pose -> watch_pose a distintas velocidades
# ---------------------------------------------------------------
velocidades = [20, 40, 60]

print('=' * 55)
print('CALIBRACIÓN DE VELOCIDADES')
print('=' * 55)
print(f'{"Velocidad":>10}  {"T real (s)":>12}  {"Estado":>10}')
print('-' * 55)

resultados_vel = []

for vel in velocidades:
    mc.send_angles(POSES['init_pose'], 50)
    time.sleep(3.0)

    t0 = time.time()
    try:
        mc.send_angles(POSES['watch_pose'], vel)
        time.sleep(0.5)
        # Esperar hasta que el robot no se mueva (is_moving devuelve 0)
        for _ in range(30):
            if mc.is_moving() == 0:
                break
            time.sleep(0.2)
        t_real = time.time() - t0
        estado = 'OK'
    except Exception as e:
        t_real = time.time() - t0
        estado = f'ERROR: {e}'

    resultados_vel.append((vel, t_real, estado))
    print(f'{vel:>10}  {t_real:>12.2f}  {estado:>10}')

print('=' * 55)

# Elegir velocidad óptima (mayor velocidad con estado OK)
vel_optima = max([v for v, t, e in resultados_vel if e == 'OK'], default=40)
MOVE_SPEED = vel_optima
print(f'\nVelocidad seleccionada: {MOVE_SPEED}')

mc.send_angles(POSES['init_pose'], 50)
time.sleep(3)

## 4. Ciclo de agarre: pick y place

**Secuencia pick:**  
`abrir gripper → ir a upper → bajar Z → cerrar gripper → subir Z`

**Secuencia place:**  
`waypoint seguro → ir a depósito → abrir gripper → subir Z`

In [ ]:

# ---------------------------------------------------------------
# Sección 2: Ciclo de agarre A → B
# Secuencia extraída del ctrl_joints.ipynb del JetCobot real
# ---------------------------------------------------------------

# Secuencia completa de un ciclo pick & place A → B
CICLO_POSES = [
    HOME,           # 1. posición de reposo
    PICK_UPPER,     # 2. moverse sobre el objeto A
    PICK_GRASP,     # 3. bajar hasta el objeto A
    'close',        # 4. cerrar gripper (agarrar)
    PICK_UPPER,     # 5. subir (sobre A con objeto)
    HOME,           # 6. pasar por home (clearance)
    PLACE_UPPER,    # 7. moverse sobre la posición B
    PLACE_GRASP,    # 8. bajar hasta B
    'open',         # 9. abrir gripper (soltar)
    PLACE_UPPER,    # 10. subir (sobre B sin objeto)
    HOME,           # 11. volver a reposo
]


def pick():
    """
    Subroutine de agarre: va de home a A, agarra y sube.
    Retorna True si completó sin excepción.
    """
    try:
        sub = [PICK_UPPER, PICK_GRASP, 'close', PICK_UPPER]
        ejecutar_secuencia(sub)
        return True
    except Exception as e:
        print(f'  [PICK] ERROR: {e}')
        return False


def place():
    """
    Subroutine de depósito: va de home a B, suelta y sube.
    Retorna True si completó sin excepción.
    """
    try:
        sub = [HOME, PLACE_UPPER, PLACE_GRASP, 'open', PLACE_UPPER, HOME]
        ejecutar_secuencia(sub)
        return True
    except Exception as e:
        print(f'  [PLACE] ERROR: {e}')
        return False


print('pick() y place() definidos.')
print(f'Pasos por ciclo completo: {len(CICLO_POSES)}')


### 4.1 Prueba unitaria de pick y place (1 ciclo manual)

In [ ]:

# ---------------------------------------------------------------
# Prueba unitaria: 1 ciclo completo antes de los 5 consecutivos
# ---------------------------------------------------------------
print('=== PRUEBA UNITARIA (1 ciclo) ===')

# Inicio: home + abrir gripper
mc.send_angles(HOME, MOVE_SPEED)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

t0 = time.time()
try:
    ejecutar_secuencia(CICLO_POSES)
    t_total = time.time() - t0
    print(f'\nPrueba unitaria OK — tiempo: {t_total:.1f}s')
except Exception as e:
    print(f'\nPrueba unitaria FALLO: {e}')
    mc.send_angles(HOME, 50)


## 5. Ejecución de 5 ciclos consecutivos

Registra el resultado de cada fase por ciclo y calcula la tasa de éxito final.

In [ ]:

# ---------------------------------------------------------------
# Sección 4: 5 ciclos consecutivos sin intervención humana
# ---------------------------------------------------------------
N_CICLOS = 5
log_ciclos = []

# Posición inicial segura
mc.send_angles(HOME, MOVE_SPEED)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

print('=' * 60)
print(f'INICIO: {N_CICLOS} ciclos  —  {datetime.now().strftime("%H:%M:%S")}')
print('=' * 60)

for ciclo in range(1, N_CICLOS + 1):
    t_inicio = time.time()
    fases = {'pick': False, 'place': False}
    exito = False

    print(f'\n--- Ciclo {ciclo}/{N_CICLOS} ---')

    # Fase pick: home → sobre A → bajar → cerrar → subir
    print(f'  Fase pick:')
    fases['pick'] = pick()

    if not fases['pick']:
        print(f'  Ciclo {ciclo}: ABORTADO en pick')
        mc.send_angles(HOME, 50)
        time.sleep(2)
        log_ciclos.append({'ciclo': ciclo, 'exito': False,
                           'tiempo': time.time() - t_inicio, 'fases': fases})
        continue

    # Fase place: home → sobre B → bajar → abrir → subir → home
    print(f'  Fase place:')
    fases['place'] = place()

    exito = fases['pick'] and fases['place']
    t_total = time.time() - t_inicio
    log_ciclos.append({'ciclo': ciclo, 'exito': exito,
                       'tiempo': t_total, 'fases': fases})

    estado = 'OK' if exito else 'FALLO'
    print(f'  Ciclo {ciclo}: {estado}  ({t_total:.1f}s)')

    if not exito:
        mc.send_angles(HOME, 50)
        time.sleep(3)

print('\n' + '=' * 60)
print('FIN DE CICLOS')
print('=' * 60)


## 6. Reporte de métricas

In [ ]:
# ---------------------------------------------------------------
# Tabla de resultados y tasa de éxito
# ---------------------------------------------------------------
n_ok  = sum(1 for r in log_ciclos if r['exito'])
tasa  = n_ok / len(log_ciclos) * 100 if log_ciclos else 0
tiempos = [r['tiempo'] for r in log_ciclos]
t_prom = sum(tiempos) / len(tiempos) if tiempos else 0

print('=' * 70)
print('REPORTE FINAL — 5 CICLOS — MyCobot 280')
print('=' * 70)
print(f'{'Ciclo':<8} {'Éxito':<8} {'Tiempo(s)':<12} {'Init':<8} {'Watch':<8} {'Pick':<8} {'Place':<8}')
print('-' * 70)
for r in log_ciclos:
    f = r['fases']
    estado = lambda v: 'OK' if v else 'FALLO'
    print(f'{r["ciclo"]:<8} {"SI" if r["exito"] else "NO":<8} '
          f'{r["tiempo"]:<12.1f} '
          f'{estado(f["init"]):<8} {estado(f["watch"]):<8} '
          f'{estado(f["pick"]):<8} {estado(f["place"]):<8}')
print('=' * 70)
print(f'Ciclos exitosos   : {n_ok}/{len(log_ciclos)}  ({tasa:.0f}%)')
print(f'Tiempo promedio   : {t_prom:.1f} s/ciclo')
print()
veredicto = 'APROBADO' if tasa >= 80 else 'REQUIERE AJUSTE'
print(f'Resultado: {veredicto}  (criterio: >= 80%)')
print('=' * 70)